# Underwater Object Detection Pipeline

This notebook combines:
1. **Download** - Fetch dataset from Harvard Dataverse
2. **Extract** - Handle 7z split archives
3. **CLIP Filtering** - Find images likely to contain objects (not empty sand)
4. **YOLO Detection** - Detect and localize actual objects
5. **Ranking** - Final scoring based on detected objects

Dataset: Harvard Dataverse DOI: doi:10.7910/DVN/VZD5S6

## 1. Setup & Imports

In [1]:
import os
import re
import json
import shutil
import heapq
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Set
import warnings
from tqdm import tqdm
warnings.filterwarnings("ignore")

import numpy as np
import requests
from PIL import Image, ImageFile, ImageDraw, ImageFont
import torch
from tqdm.notebook import tqdm
import cv2
from scipy.ndimage import convolve

# CLIP for semantic filtering
from transformers import CLIPProcessor, CLIPModel

# YOLO for object detection
from ultralytics import YOLO

# 7z extraction
import py7zr

ImageFile.LOAD_TRUNCATED_IMAGES = True

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.10.0+cu128
CUDA available: False


## 2. Configuration

In [2]:

# Dataset
DATASET_DOI = "doi:10.7910/DVN/VZD5S6"
DATAVERSE_BASE_URL = "https://dataverse.harvard.edu"

# Directories
OUTPUT_DIR = Path("./ranked_dataset")
TEMP_DIR = OUTPUT_DIR / "temp"
IMAGES_DIR = OUTPUT_DIR / "images"
RESULTS_DIR = OUTPUT_DIR / "results"

# Create directories
for d in [OUTPUT_DIR, TEMP_DIR, IMAGES_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Processing parameters
TOP_K = 200                    # Number of images to keep
CLIP_BATCH_SIZE = 16           # Batch size for CLIP
YOLO_CONFIDENCE = 0.25         # YOLO detection confidence threshold
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# File extensions
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}

print(f"Output directory: {OUTPUT_DIR}")
print(f"Device: {DEVICE}")
print(f"Top-K: {TOP_K}")

Output directory: ranked_dataset
Device: cpu
Top-K: 200


## 3. CLIP Prompts for Object Detection

These prompts help CLIP identify images that likely contain **objects of interest** rather than empty seafloor.

In [3]:
# Positive prompts - looking for OBJECTS on seafloor
POSITIVE_PROMPTS = [
    # Shipwrecks and vessels
    "sonar image of a shipwreck on the ocean floor",
    "underwater photo of a sunken ship or boat wreckage",
    "side-scan sonar showing a vessel or ship debris on seabed",
    
    # Man-made objects and structures
    "sonar image showing man-made objects on the seafloor",
    "underwater image of debris, wreckage, or artifacts on ocean bottom",
    "acoustic image of underwater infrastructure, pipes, or cables",
    "sonar scan of an anchor, chain, or maritime equipment",
    
    # Vehicles and aircraft
    "sonar image of a submerged vehicle or aircraft wreckage",
    "underwater photo of a car, plane, or machinery on seabed",
    
    # General objects of interest
    "side-scan sonar with distinct object casting shadow on seafloor",
    "underwater image showing anomaly or unusual object on ocean floor",
    "sonar image with clear man-made structure or debris field",
]

# Negative prompts - penalize empty/boring images
NEGATIVE_PROMPTS = [
    # Empty seafloor
    "empty sandy seafloor with no objects or features",
    "flat featureless ocean bottom with just sand and sediment",
    "uniform seabed texture with nothing interesting",
    "boring empty underwater scene with only sand",
    
    # Technical issues
    "completely black empty image with no content",
    "pure white overexposed blank image",
    "random noise static corruption artifacts",
    "extremely blurry out of focus underwater image",
]

# Scoring weights - prioritize semantic content over technical quality
CLIP_WEIGHTS = {
    "clip_positive": 0.50,    # HIGH - "does it have objects?"
    "clip_negative": 0.20,    # "is it NOT empty sand?"
    "entropy": 0.05,          # LOW - technical quality less important
    "laplacian_var": 0.05,    # LOW - slightly blurry shipwreck still valuable
    "saturation_penalty": 0.10,
    "edge_density": 0.10,     # edges might indicate object boundaries
}

print(f"Positive prompts: {len(POSITIVE_PROMPTS)}")
print(f"Negative prompts: {len(NEGATIVE_PROMPTS)}")

Positive prompts: 12
Negative prompts: 8


## 4. Harvard Dataverse Download Functions

## Network Configuration (If Behind Firewall/Proxy)

If you're behind a corporate firewall or proxy, configure it here:

In [ ]:
def get_dataset_metadata(doi: str) -> Dict:
    """Fetch dataset metadata from Dataverse API."""
    url = f"{DATAVERSE_BASE_URL}/api/datasets/:persistentId/?persistentId={doi}"
    print(f"Fetching metadata for {doi}...")
    print(f"URL: {url}")
    
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
    print(f"Status code: {response.status_code}")
    print(f"Content-Type: {response.headers.get('content-type', 'N/A')}")
    
    if response.status_code != 200:
        raise RuntimeError(f"Failed: {response.status_code} - {response.text}")
    
    # Debug: print first 200 chars of response
    print(f"Response preview: {response.text[:200]}")
    
    # Check if response is empty
    if not response.text or response.text.strip() == "":
        raise RuntimeError(f"Empty response from API. This might be a network issue or invalid DOI.")
    
    # Check if it's JSON
    content_type = response.headers.get('content-type', '')
    if 'json' not in content_type.lower():
        print(f"WARNING: Response is not JSON (content-type: {content_type})")
        print(f"First 500 chars: {response.text[:500]}")
        raise RuntimeError(f"API returned non-JSON response. This might be an HTML error page.")
    
    return response.json()


def download_file(file_id: int, filename: str, download_path: Path) -> bool:
    """Download a single file from Dataverse with progress bar."""
    url = f"{DATAVERSE_BASE_URL}/api/access/datafile/{file_id}"
    
    response = requests.get(url, stream=True)
    if response.status_code != 200:
        print(f"Failed to download {filename}: {response.status_code}")
        return False
    
    total_size = int(response.headers.get('content-length', 0))
    download_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(download_path, 'wb') as f:
        with tqdm(total=total_size, unit='iB', unit_scale=True, desc=filename) as pbar:
            for chunk in response.iter_content(chunk_size=8192):
                size = f.write(chunk)
                pbar.update(size)
    
    return True


def list_dataset_files(doi: str) -> List[Dict]:
    """List all files in the dataset."""
    metadata = get_dataset_metadata(doi)
    files = metadata['data']['latestVersion']['files']
    
    result = []
    for file_info in files:
        datafile = file_info.get('dataFile', {})
        result.append({
            'id': datafile.get('id'),
            'filename': datafile.get('filename', ''),
            'size': datafile.get('filesize', 0),
        })
    
    return result


# Test connection
try:
    files = list_dataset_files(DATASET_DOI)
    print(f"\nFound {len(files)} files in dataset:")
    total_size = sum(f['size'] for f in files)
    print(f"Total size: {total_size / (1024**3):.2f} GB")
    for f in files[:5]:
        print(f"  - {f['filename']} ({f['size']/(1024**2):.1f} MB)")
    if len(files) > 5:
        print(f"  ... and {len(files)-5} more")
except Exception as e:
    print(f"Error connecting to Dataverse: {e}")
    print(f"\nPossible solutions:")
    print(f"  1. Check your internet connection")
    print(f"  2. Verify the DOI is correct: {DATASET_DOI}")
    print(f"  3. Try accessing the dataset directly in browser:")
    print(f"     https://dataverse.harvard.edu/dataset.xhtml?persistentId={DATASET_DOI}")
    print(f"  4. The dataset might be restricted or moved")

## 5. 7z Archive Extraction

In [ ]:
def is_split_7z_part(filename: str) -> bool:
    """Check if file is part of a split 7z archive."""
    return bool(re.search(r'\.7z\.\d{3}$', filename.lower()))


def get_split_7z_base(filename: str) -> str:
    """Get base name of split 7z archive."""
    return re.sub(r'\.\d{3}$', '', filename)


def concatenate_split_archive(parts: List[Path], output_path: Path) -> bool:
    """Concatenate split 7z archive parts into a single file."""
    try:
        print(f"  Concatenating {len(parts)} parts into {output_path.name}...")
        with open(output_path, 'wb') as outfile:
            for part in sorted(parts):
                print(f"    Adding {part.name}...")
                with open(part, 'rb') as infile:
                    shutil.copyfileobj(infile, outfile)
        return True
    except Exception as e:
        print(f"  Concatenation failed: {e}")
        return False


def extract_7z_archive(archive_path: Path, extract_dir: Path, is_split: bool = False, all_parts: List[Path] = None) -> List[Path]:
    """
    Extract 7z archive and return list of image paths.
    
    For split archives, tries multiple methods:
    1. py7zr (if concatenated)
    2. System 7z command
    3. Concatenate then extract
    """
    extracted_images = []
    
    # Method 1: Try system 7z command first (handles split archives natively)
    try:
        import subprocess
        print(f"  Trying system 7z extraction...")
        result = subprocess.run(
            ['7z', 'x', str(archive_path), f'-o{extract_dir}', '-y'],
            capture_output=True, text=True, timeout=3600
        )
        if result.returncode == 0:
            print(f"  System 7z extraction successful!")
        else:
            raise Exception(f"7z returned {result.returncode}: {result.stderr}")
    except FileNotFoundError:
        print(f"  System 7z not found, trying py7zr...")
        # Method 2: Try py7zr
        try:
            # For split archives, we need to concatenate first
            if is_split and all_parts and len(all_parts) > 1:
                combined_path = archive_path.parent / archive_path.name.replace('.001', '_combined.7z')
                if concatenate_split_archive(all_parts, combined_path):
                    archive_path = combined_path
            
            print(f"  Extracting {archive_path.name} with py7zr...")
            with py7zr.SevenZipFile(archive_path, mode='r') as z:
                all_files = z.getnames()
                print(f"  Archive contains {len(all_files)} files")
                z.extractall(path=extract_dir)
                
        except Exception as e:
            print(f"  py7zr extraction failed: {e}")
            print(f"  Tip: Install p7zip-full: sudo apt-get install p7zip-full")
            return []
    except Exception as e:
        print(f"  System 7z extraction failed: {e}")
        return []
    
    # Find all extracted images
    for root, dirs, files in os.walk(extract_dir):
        for f in files:
            ext = os.path.splitext(f)[1].lower()
            if ext in IMAGE_EXTENSIONS:
                extracted_images.append(Path(root) / f)
    
    print(f"  Found {len(extracted_images)} images")
    return extracted_images


def extract_zip_archive(archive_path: Path, extract_dir: Path) -> List[Path]:
    """Extract zip archive and return list of image paths."""
    import zipfile
    extracted_images = []
    
    try:
        print(f"  Extracting {archive_path.name}...")
        with zipfile.ZipFile(archive_path, 'r') as z:
            z.extractall(extract_dir)
        
        for root, dirs, files in os.walk(extract_dir):
            for f in files:
                ext = os.path.splitext(f)[1].lower()
                if ext in IMAGE_EXTENSIONS:
                    extracted_images.append(Path(root) / f)
        
        print(f"  Found {len(extracted_images)} images")
    except Exception as e:
        print(f"  Extraction failed: {e}")
    
    return extracted_images


def extract_tar_archive(archive_path: Path, extract_dir: Path) -> List[Path]:
    """Extract tar/tar.gz/tar.bz2 archive and return list of image paths."""
    import tarfile
    extracted_images = []
    
    try:
        print(f"  Extracting {archive_path.name}...")
        with tarfile.open(archive_path, 'r:*') as t:
            t.extractall(extract_dir)
        
        for root, dirs, files in os.walk(extract_dir):
            for f in files:
                ext = os.path.splitext(f)[1].lower()
                if ext in IMAGE_EXTENSIONS:
                    extracted_images.append(Path(root) / f)
        
        print(f"  Found {len(extracted_images)} images")
    except Exception as e:
        print(f"  Extraction failed: {e}")
    
    return extracted_images


def extract_any_archive(archive_path: Path, extract_dir: Path, all_parts: List[Path] = None) -> List[Path]:
    """
    Smart extraction that handles multiple archive formats.
    Supports: .7z, .zip, .tar, .tar.gz, .tgz, .tar.bz2
    """
    name = archive_path.name.lower()
    
    if '.7z' in name:
        is_split = bool(re.search(r'\.7z\.\d{3}$', name))
        return extract_7z_archive(archive_path, extract_dir, is_split=is_split, all_parts=all_parts)
    elif name.endswith('.zip'):
        return extract_zip_archive(archive_path, extract_dir)
    elif name.endswith(('.tar', '.tar.gz', '.tgz', '.tar.bz2')):
        return extract_tar_archive(archive_path, extract_dir)
    else:
        print(f"  Unknown archive format: {name}")
        return []


def group_split_archives(files: List[Dict]) -> Dict[str, List[Dict]]:
    """Group split 7z archives by base name."""
    groups = {}
    
    for f in files:
        filename = f['filename']
        if is_split_7z_part(filename):
            base = get_split_7z_base(filename)
            if base not in groups:
                groups[base] = []
            groups[base].append(f)
    
    # Sort parts within each group
    for base in groups:
        groups[base] = sorted(groups[base], key=lambda x: x['filename'])
    
    return groups


# Test: check if 7z is installed
try:
    import subprocess
    result = subprocess.run(['7z', '--help'], capture_output=True)
    print("✓ System 7z is available (recommended for split archives)")
except FileNotFoundError:
    print("✗ System 7z not found. Install with: sudo apt-get install p7zip-full")
    print("  Will fall back to py7zr (may have issues with split archives)")

## 6. CLIP Scorer for Semantic Filtering

In [ ]:
class CLIPScorer:
    """CLIP-based image scoring with precomputed text embeddings."""
    
    def __init__(self, model_name: str = "openai/clip-vit-base-patch32", device: str = "cpu"):
        self.device = device
        print(f"Loading CLIP model: {model_name} on {device}")
        
        self.model = CLIPModel.from_pretrained(model_name)
        self.processor = CLIPProcessor.from_pretrained(model_name)
        self.model.to(device)
        self.model.eval()
        
        # Precompute text embeddings
        with torch.no_grad():
            # Positive prompts
            pos_inputs = self.processor(text=POSITIVE_PROMPTS, return_tensors="pt", padding=True, truncation=True)
            pos_inputs = {k: v.to(device) for k, v in pos_inputs.items() if k != "pixel_values"}
            pos_output = self.model.get_text_features(**pos_inputs)
            # Handle both tensor and BaseModelOutputWithPooling returns (transformers compatibility)
            self.pos_embeds = pos_output if isinstance(pos_output, torch.Tensor) else pos_output.pooler_output
            self.pos_embeds = self.pos_embeds / self.pos_embeds.norm(dim=-1, keepdim=True)
            
            # Negative prompts
            neg_inputs = self.processor(text=NEGATIVE_PROMPTS, return_tensors="pt", padding=True, truncation=True)
            neg_inputs = {k: v.to(device) for k, v in neg_inputs.items() if k != "pixel_values"}
            neg_output = self.model.get_text_features(**neg_inputs)
            # Handle both tensor and BaseModelOutputWithPooling returns (transformers compatibility)
            self.neg_embeds = neg_output if isinstance(neg_output, torch.Tensor) else neg_output.pooler_output
            self.neg_embeds = self.neg_embeds / self.neg_embeds.norm(dim=-1, keepdim=True)
        
        print(f"CLIP ready with {len(POSITIVE_PROMPTS)} positive, {len(NEGATIVE_PROMPTS)} negative prompts")
    
    def score_batch(self, images: List[Image.Image]) -> List[Dict[str, float]]:
        """Score a batch of PIL images."""
        if not images:
            return []
        
        inputs = self.processor(images=images, return_tensors="pt", padding=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            img_output = self.model.get_image_features(**inputs)
            # Handle both tensor and BaseModelOutputWithPooling returns (transformers compatibility)
            img_embeds = img_output if isinstance(img_output, torch.Tensor) else img_output.pooler_output
            img_embeds = img_embeds / img_embeds.norm(dim=-1, keepdim=True)
            
            pos_sim = torch.matmul(img_embeds, self.pos_embeds.T).mean(dim=-1)
            neg_sim = torch.matmul(img_embeds, self.neg_embeds.T).mean(dim=-1)
        
        pos_scores = ((pos_sim.cpu().numpy() + 1) / 2).clip(0, 1)
        neg_scores = 1.0 - ((neg_sim.cpu().numpy() + 1) / 2).clip(0, 1)
        
        return [{"clip_positive": float(p), "clip_negative": float(n)} for p, n in zip(pos_scores, neg_scores)]


# Initialize CLIP scorer
clip_scorer = CLIPScorer(device=DEVICE)

Loading CLIP model: openai/clip-vit-base-patch32 on cuda


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


CLIP ready with 12 positive, 8 negative prompts


## 7. YOLO Object Detector

In [ ]:
class YOLODetector:
    """
    YOLO-based object detector.
    
    Uses YOLOv8 pretrained on COCO dataset.
    For underwater-specific detection, you could fine-tune on:
    - FathomNet dataset
    - Custom underwater object dataset
    """
    
    # Object classes of interest for underwater scenes
    INTERESTING_CLASSES = {
        # Vehicles
        'car', 'truck', 'bus', 'motorcycle', 'bicycle',
        'boat', 'airplane', 'train',
        # People
        'person',
        # Objects
        'backpack', 'umbrella', 'handbag', 'suitcase',
        'bottle', 'cup', 'chair', 'couch', 'bed',
        'refrigerator', 'toilet', 'tv', 'laptop', 'cell phone',
        # Animals (marine life proxy)
        'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
        'elephant', 'bear', 'zebra', 'giraffe',
        # Sports/recreation
        'sports ball', 'kite', 'baseball bat', 'skateboard', 'surfboard',
        'tennis racket', 'frisbee', 'skis', 'snowboard',
    }
    
    def __init__(self, model_name: str = "yolov8n.pt", confidence: float = 0.25):
        """
        Initialize YOLO detector.
        
        Models (speed vs accuracy):
        - yolov8n.pt: Nano (fastest, least accurate)
        - yolov8s.pt: Small
        - yolov8m.pt: Medium
        - yolov8l.pt: Large
        - yolov8x.pt: XLarge (slowest, most accurate)
        """
        print(f"Loading YOLO model: {model_name}")
        self.model = YOLO(model_name)
        self.confidence = confidence
        self.class_names = self.model.names
        print(f"YOLO ready with {len(self.class_names)} classes")
    
    def detect(self, image: Image.Image) -> Dict:
        """
        Run detection on a single image.
        
        Returns:
            {
                'num_detections': int,
                'num_interesting': int,
                'detections': [{'class': str, 'confidence': float, 'bbox': [x1,y1,x2,y2]}],
                'interesting_detections': [...],
                'detection_score': float  # 0-1 score based on detections
            }
        """
        # Run inference
        results = self.model(image, conf=self.confidence, verbose=False)
        
        detections = []
        interesting_detections = []
        
        for result in results:
            boxes = result.boxes
            if boxes is not None:
                for i in range(len(boxes)):
                    cls_id = int(boxes.cls[i].item())
                    cls_name = self.class_names[cls_id]
                    conf = float(boxes.conf[i].item())
                    bbox = boxes.xyxy[i].tolist()
                    
                    det = {
                        'class': cls_name,
                        'confidence': conf,
                        'bbox': bbox
                    }
                    detections.append(det)
                    
                    if cls_name in self.INTERESTING_CLASSES:
                        interesting_detections.append(det)
        
        # Calculate detection score
        # More detections and higher confidence = higher score
        if interesting_detections:
            # Weight by confidence
            conf_sum = sum(d['confidence'] for d in interesting_detections)
            detection_score = min(1.0, conf_sum / 2.0)  # Cap at 1.0
        elif detections:
            conf_sum = sum(d['confidence'] for d in detections)
            detection_score = min(0.5, conf_sum / 4.0)  # Cap at 0.5 for non-interesting
        else:
            detection_score = 0.0
        
        return {
            'num_detections': len(detections),
            'num_interesting': len(interesting_detections),
            'detections': detections,
            'interesting_detections': interesting_detections,
            'detection_score': detection_score
        }
    
    def detect_batch(self, images: List[Image.Image]) -> List[Dict]:
        """Run detection on a batch of images."""
        return [self.detect(img) for img in images]
    
    def draw_detections(self, image: Image.Image, detections: List[Dict]) -> Image.Image:
        """Draw bounding boxes on image."""
        img_draw = image.copy()
        draw = ImageDraw.Draw(img_draw)
        
        for det in detections:
            bbox = det['bbox']
            cls_name = det['class']
            conf = det['confidence']
            
            # Color: green for interesting, yellow for others
            color = (0, 255, 0) if cls_name in self.INTERESTING_CLASSES else (255, 255, 0)
            
            draw.rectangle(bbox, outline=color, width=2)
            draw.text((bbox[0], bbox[1]-15), f"{cls_name} {conf:.2f}", fill=color)
        
        return img_draw


# Initialize YOLO detector
yolo_detector = YOLODetector(model_name="yolov8m.pt", confidence=YOLO_CONFIDENCE)

Loading YOLO model: yolov8m.pt
YOLO ready with 80 classes


## 8. CV Heuristics (Technical Quality)

In [ ]:
def compute_grayscale_entropy(img_array: np.ndarray) -> float:
    """Compute Shannon entropy of grayscale histogram."""
    if img_array.ndim == 3:
        gray = np.dot(img_array[..., :3], [0.2989, 0.5870, 0.1140]).astype(np.uint8)
    else:
        gray = img_array
    
    hist, _ = np.histogram(gray.flatten(), bins=256, range=(0, 256))
    hist = hist.astype(np.float64)
    hist = hist[hist > 0]
    hist = hist / hist.sum()
    entropy = -np.sum(hist * np.log2(hist))
    return min(entropy / 8.0, 1.0)


def compute_laplacian_variance(img_array: np.ndarray) -> float:
    """Compute Laplacian variance (sharpness)."""
    if img_array.ndim == 3:
        gray = np.dot(img_array[..., :3], [0.2989, 0.5870, 0.1140]).astype(np.float32)
    else:
        gray = img_array.astype(np.float32)
    
    kernel = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32)
    laplacian = convolve(gray, kernel, mode='reflect')
    variance = laplacian.var()
    normalized = 2.0 / (1.0 + np.exp(-variance / 500.0)) - 1.0
    return float(np.clip(normalized, 0, 1))


def compute_saturation_penalty(img_array: np.ndarray) -> float:
    """Compute penalty for saturated pixels."""
    if img_array.ndim == 3:
        gray = np.dot(img_array[..., :3], [0.2989, 0.5870, 0.1140])
    else:
        gray = img_array.astype(np.float64)
    
    total_pixels = gray.size
    near_black = np.sum(gray < 5)
    near_white = np.sum(gray > 250)
    saturated_ratio = (near_black + near_white) / total_pixels
    return float(np.exp(-5.0 * saturated_ratio))


def compute_edge_density(img_array: np.ndarray) -> float:
    """Compute edge density using Canny."""
    if img_array.ndim == 3:
        gray = np.dot(img_array[..., :3], [0.2989, 0.5870, 0.1140]).astype(np.uint8)
    else:
        gray = img_array.astype(np.uint8)
    
    edges = cv2.Canny(gray, 50, 150)
    edge_ratio = np.sum(edges > 0) / edges.size
    return float(min(edge_ratio * 10, 1.0))


def compute_cv_heuristics(img_array: np.ndarray) -> Dict[str, float]:
    """Compute all CV heuristic scores."""
    try:
        return {
            "entropy": compute_grayscale_entropy(img_array),
            "laplacian_var": compute_laplacian_variance(img_array),
            "saturation_penalty": compute_saturation_penalty(img_array),
            "edge_density": compute_edge_density(img_array),
        }
    except Exception as e:
        return {"entropy": 0.0, "laplacian_var": 0.0, "saturation_penalty": 0.0, "edge_density": 0.0}

## 9. Combined Scoring Function

In [ ]:
@dataclass
class ImageResult:
    """Complete result for a scored image."""
    path: str
    clip_score: float
    yolo_score: float
    combined_score: float
    clip_components: Dict[str, float]
    cv_components: Dict[str, float]
    yolo_detections: List[Dict]
    num_detections: int
    num_interesting: int


def compute_combined_score(
    clip_scores: Dict[str, float],
    cv_scores: Dict[str, float],
    yolo_result: Dict,
    clip_weight: float = 0.4,
    yolo_weight: float = 0.5,
    cv_weight: float = 0.1
) -> Tuple[float, float, float]:
    """
    Compute combined score from CLIP, YOLO, and CV.
    
    Returns: (combined_score, clip_score, yolo_score)
    """
    # CLIP score (weighted sum of components)
    clip_score = (
        CLIP_WEIGHTS["clip_positive"] * clip_scores.get("clip_positive", 0) +
        CLIP_WEIGHTS["clip_negative"] * clip_scores.get("clip_negative", 0)
    ) / (CLIP_WEIGHTS["clip_positive"] + CLIP_WEIGHTS["clip_negative"])
    
    # CV score (weighted sum)
    cv_score = (
        CLIP_WEIGHTS["entropy"] * cv_scores.get("entropy", 0) +
        CLIP_WEIGHTS["laplacian_var"] * cv_scores.get("laplacian_var", 0) +
        CLIP_WEIGHTS["saturation_penalty"] * cv_scores.get("saturation_penalty", 0) +
        CLIP_WEIGHTS["edge_density"] * cv_scores.get("edge_density", 0)
    ) / (CLIP_WEIGHTS["entropy"] + CLIP_WEIGHTS["laplacian_var"] + 
         CLIP_WEIGHTS["saturation_penalty"] + CLIP_WEIGHTS["edge_density"])
    
    # YOLO score
    yolo_score = yolo_result.get("detection_score", 0)
    
    # Combined score
    combined = (
        clip_weight * clip_score +
        yolo_weight * yolo_score +
        cv_weight * cv_score
    )
    
    return float(np.clip(combined, 0, 1)), clip_score, yolo_score


def load_image_safe(path: Path, max_size: int = 1024) -> Optional[Tuple[Image.Image, np.ndarray]]:
    """Safely load and resize an image."""
    try:
        img = Image.open(path)
        if img.mode != "RGB":
            img = img.convert("RGB")
        
        if max(img.size) > max_size:
            ratio = max_size / max(img.size)
            new_size = (int(img.size[0] * ratio), int(img.size[1] * ratio))
            img = img.resize(new_size, Image.LANCZOS)
        
        arr = np.array(img)
        if arr.size == 0 or arr.ndim < 2:
            return None
        
        return img, arr
    except Exception as e:
        return None

## 10. Main Processing Pipeline

In [ ]:
def process_images(
    image_paths: List[Path],
    clip_scorer: CLIPScorer,
    yolo_detector: YOLODetector,
    top_k: int = 200,
    batch_size: int = 16
) -> List[ImageResult]:
    """
    Process images through CLIP + YOLO pipeline.
    
    Returns top-K images sorted by combined score.
    """
    # Min-heap for top-K (stores negative score for max-heap behavior)
    top_k_heap = []  # (score, index, ImageResult)
    
    # Process in batches
    batch_data = []
    
    pbar = tqdm(image_paths, desc="Processing images", unit="img")
    
    for idx, img_path in enumerate(pbar):
        result = load_image_safe(img_path)
        if result is None:
            continue
        
        pil_img, np_arr = result
        batch_data.append((idx, img_path, pil_img, np_arr))
        
        if len(batch_data) >= batch_size:
            _process_batch(batch_data, clip_scorer, yolo_detector, top_k_heap, top_k)
            batch_data = []
            
            # Update progress bar with current best score
            if top_k_heap:
                best_score = -top_k_heap[0][0] if top_k_heap else 0
                pbar.set_postfix({"best": f"{best_score:.3f}", "heap": len(top_k_heap)})
    
    # Process remaining
    if batch_data:
        _process_batch(batch_data, clip_scorer, yolo_detector, top_k_heap, top_k)
    
    # Extract results and sort by score descending
    results = [item[2] for item in top_k_heap]
    results.sort(key=lambda x: x.combined_score, reverse=True)
    
    return results


def _process_batch(
    batch_data: List[Tuple],
    clip_scorer: CLIPScorer,
    yolo_detector: YOLODetector,
    top_k_heap: List,
    top_k: int
):
    """Process a batch of images."""
    pil_images = [item[2] for item in batch_data]
    
    # CLIP scoring (batched)
    try:
        clip_scores_list = clip_scorer.score_batch(pil_images)
    except Exception as e:
        clip_scores_list = [{"clip_positive": 0.0, "clip_negative": 0.5}] * len(batch_data)
    
    # Process each image
    for i, (idx, img_path, pil_img, np_arr) in enumerate(batch_data):
        # CV heuristics
        cv_scores = compute_cv_heuristics(np_arr)
        
        # YOLO detection
        yolo_result = yolo_detector.detect(pil_img)
        
        # Combined score
        clip_scores = clip_scores_list[i] if i < len(clip_scores_list) else {"clip_positive": 0.0, "clip_negative": 0.5}
        combined, clip_score, yolo_score = compute_combined_score(clip_scores, cv_scores, yolo_result)
        
        # Create result
        result = ImageResult(
            path=str(img_path),
            clip_score=clip_score,
            yolo_score=yolo_score,
            combined_score=combined,
            clip_components=clip_scores,
            cv_components=cv_scores,
            yolo_detections=yolo_result['detections'],
            num_detections=yolo_result['num_detections'],
            num_interesting=yolo_result['num_interesting']
        )
        
        # Update heap (use negative score for min-heap to act as max-heap)
        entry = (-combined, idx, result)
        
        if len(top_k_heap) < top_k:
            heapq.heappush(top_k_heap, entry)
        elif combined > -top_k_heap[0][0]:
            heapq.heapreplace(top_k_heap, entry)

## 11. Test on Sample Images

Test the pipeline on a few sample images before running on the full dataset.

In [ ]:
# Test with a sample image (if available)
import matplotlib.pyplot as plt

def test_on_image(image_path: str):
    """Test the full pipeline on a single image."""
    result = load_image_safe(Path(image_path))
    if result is None:
        print(f"Could not load image: {image_path}")
        return
    
    pil_img, np_arr = result
    
    # CLIP score
    clip_scores = clip_scorer.score_batch([pil_img])[0]
    print(f"CLIP Scores:")
    print(f"  Positive (has objects): {clip_scores['clip_positive']:.3f}")
    print(f"  Negative (not empty):   {clip_scores['clip_negative']:.3f}")
    
    # CV scores
    cv_scores = compute_cv_heuristics(np_arr)
    print(f"\nCV Scores:")
    for k, v in cv_scores.items():
        print(f"  {k}: {v:.3f}")
    
    # YOLO detection
    yolo_result = yolo_detector.detect(pil_img)
    print(f"\nYOLO Detections:")
    print(f"  Total: {yolo_result['num_detections']}")
    print(f"  Interesting: {yolo_result['num_interesting']}")
    print(f"  Detection score: {yolo_result['detection_score']:.3f}")
    
    for det in yolo_result['detections'][:5]:
        print(f"    - {det['class']}: {det['confidence']:.2f}")
    
    # Combined score
    combined, clip_final, yolo_final = compute_combined_score(clip_scores, cv_scores, yolo_result)
    print(f"\nFinal Scores:")
    print(f"  CLIP:     {clip_final:.3f}")
    print(f"  YOLO:     {yolo_final:.3f}")
    print(f"  Combined: {combined:.3f}")
    
    # Display image with detections
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    axes[0].imshow(pil_img)
    axes[0].set_title("Original")
    axes[0].axis('off')
    
    img_with_boxes = yolo_detector.draw_detections(pil_img, yolo_result['detections'])
    axes[1].imshow(img_with_boxes)
    axes[1].set_title(f"YOLO Detections ({yolo_result['num_detections']} found)")
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

# Uncomment to test on a specific image:
# test_on_image("path/to/your/image.jpg")

## 12. Download and Process Dataset

**WARNING**: The camera.7z archive is ~128GB. Make sure you have enough disk space.

In [ ]:
def download_and_process_archive(
    archive_name: str,
    parts: List[Dict],
    clip_scorer: CLIPScorer,
    yolo_detector: YOLODetector,
    top_k: int = 200
) -> List[ImageResult]:
    """
    Download all parts of a split archive, extract, and process images.
    """
    print(f"\n{'='*60}")
    print(f"Processing: {archive_name}")
    print(f"Parts: {len(parts)}")
    total_size = sum(p['size'] for p in parts)
    print(f"Total size: {total_size/(1024**3):.2f} GB")
    print(f"{'='*60}")
    
    # Download all parts
    print(f"\n[Step 1] Downloading {len(parts)} parts...")
    downloaded_parts = []
    
    for i, part in enumerate(parts):
        part_path = TEMP_DIR / part['filename']
        print(f"\nPart {i+1}/{len(parts)}: {part['filename']}")
        
        if part_path.exists():
            print(f"  Already exists, skipping download")
        else:
            if not download_file(part['id'], part['filename'], part_path):
                print(f"  FAILED!")
                return []
        
        downloaded_parts.append(part_path)
    
    # Extract
    print(f"\n[Step 2] Extracting archive...")
    extract_dir = TEMP_DIR / f"extract_{archive_name.replace('.7z', '')}"
    extract_dir.mkdir(exist_ok=True)
    
    # Use the improved extraction function with all parts
    image_paths = extract_any_archive(
        downloaded_parts[0], 
        extract_dir, 
        all_parts=downloaded_parts
    )
    
    if not image_paths:
        print("  No images found!")
        return []
    
    # Process images
    print(f"\n[Step 3] Processing {len(image_paths)} images with CLIP + YOLO...")
    results = process_images(image_paths, clip_scorer, yolo_detector, top_k=top_k)
    
    # Cleanup (optional - comment out to keep files)
    # print(f"\n[Step 4] Cleaning up...")
    # shutil.rmtree(extract_dir, ignore_errors=True)
    # for p in downloaded_parts:
    #     if p.exists():
    #         p.unlink()
    
    print(f"\nComplete! Found {len(results)} top images.")
    return results


def process_local_archive(
    archive_path: Path,
    clip_scorer: CLIPScorer,
    yolo_detector: YOLODetector,
    top_k: int = 200
) -> List[ImageResult]:
    """
    Process a local archive file (7z, zip, tar, etc).
    Use this when you've already downloaded the archive.
    """
    print(f"\n{'='*60}")
    print(f"Processing local archive: {archive_path.name}")
    print(f"{'='*60}")
    
    if not archive_path.exists():
        print(f"Error: Archive not found: {archive_path}")
        return []
    
    # Find all parts if this is a split archive
    all_parts = None
    if is_split_7z_part(archive_path.name):
        base = get_split_7z_base(archive_path.name)
        all_parts = sorted(archive_path.parent.glob(f"{base}.*"))
        print(f"Found {len(all_parts)} parts for split archive")
    
    # Extract
    print(f"\n[Step 1] Extracting archive...")
    extract_dir = archive_path.parent / f"extract_{archive_path.stem}"
    extract_dir.mkdir(exist_ok=True)
    
    image_paths = extract_any_archive(archive_path, extract_dir, all_parts=all_parts)
    
    if not image_paths:
        print("  No images found!")
        return []
    
    # Process images
    print(f"\n[Step 2] Processing {len(image_paths)} images with CLIP + YOLO...")
    results = process_images(image_paths, clip_scorer, yolo_detector, top_k=top_k)
    
    print(f"\nComplete! Found {len(results)} top images.")
    return results

In [ ]:
# Get dataset files
all_files = list_dataset_files(DATASET_DOI)
archive_groups = group_split_archives(all_files)

print("Available archives:")
for name, parts in archive_groups.items():
    size_gb = sum(p['size'] for p in parts) / (1024**3)
    print(f"  {name}: {len(parts)} parts, {size_gb:.2f} GB")

Fetching metadata for doi:10.7910/DVN/VZD5S6...
Available archives:
  camera.7z: 53 parts, 128.54 GB
  sonar.7z: 1 parts, 0.06 GB


In [ ]:
sonar_results = download_and_process_archive(
    "sonar.7z",
    archive_groups["sonar.7z"],
    clip_scorer,
    yolo_detector,
    top_k=TOP_K
)


Processing: sonar.7z
Parts: 1
Total size: 0.06 GB

[Step 1] Downloading 1 parts...

Part 1/1: sonar.7z.001
  Already exists, skipping download

[Step 2] Extracting archive...
  Extracting sonar.7z.001...
  Extraction failed: not a 7z file
  No images found!


In [ ]:
# Process the camera archive (LARGE: ~128GB)
# Make sure you have enough disk space!
# Uncomment to run:

camera_results = download_and_process_archive(
    "camera.7z",
    archive_groups["camera.7z"],
    clip_scorer,
    yolo_detector,
    top_k=TOP_K
)


Processing: camera.7z
Parts: 53
Total size: 128.54 GB

[Step 1] Downloading 53 parts...

Part 1/53: camera.7z.001
  Already exists, skipping download

Part 2/53: camera.7z.002
  Already exists, skipping download

Part 3/53: camera.7z.003


camera.7z.003:   0%|          | 0.00/2.62G [00:00<?, ?iB/s]


Part 4/53: camera.7z.004


camera.7z.004:   0%|          | 0.00/2.62G [00:00<?, ?iB/s]


Part 5/53: camera.7z.005


...


Part 52/53: camera.7z.052


camera.7z.052:   0%|          | 0.00/2.62G [00:00<?, ?iB/s]


Part 53/53: camera.7z.053


camera.7z.053:   0%|          | 0.00/1.71G [00:00<?, ?iB/s]


[Step 2] Extracting archive...
  Extracting camera.7z.001...
  Extraction failed: not a 7z file
  No images found!


## 13. Save and Visualize Results

In [ ]:
def save_results(results: List[ImageResult], output_dir: Path, prefix: str = "results"):
    """Save results to JSON and CSV."""
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # JSON
    json_data = []
    for i, r in enumerate(results, 1):
        json_data.append({
            "rank": i,
            "path": r.path,
            "combined_score": r.combined_score,
            "clip_score": r.clip_score,
            "yolo_score": r.yolo_score,
            "num_detections": r.num_detections,
            "num_interesting": r.num_interesting,
            "clip_components": r.clip_components,
            "cv_components": r.cv_components,
            "yolo_detections": r.yolo_detections
        })
    
    json_path = output_dir / f"{prefix}.json"
    with open(json_path, "w") as f:
        json.dump(json_data, f, indent=2)
    print(f"Saved: {json_path}")
    
    # CSV (summary)
    csv_path = output_dir / f"{prefix}.csv"
    with open(csv_path, "w") as f:
        f.write("rank,path,combined_score,clip_score,yolo_score,num_detections,num_interesting\n")
        for i, r in enumerate(results, 1):
            f.write(f"{i},{r.path},{r.combined_score:.4f},{r.clip_score:.4f},{r.yolo_score:.4f},{r.num_detections},{r.num_interesting}\n")
    print(f"Saved: {csv_path}")


def visualize_top_results(results: List[ImageResult], n: int = 10):
    """Visualize top N results with YOLO detections."""
    n = min(n, len(results))
    
    fig, axes = plt.subplots(n, 2, figsize=(14, 5*n))
    if n == 1:
        axes = [axes]
    
    for i, result in enumerate(results[:n]):
        try:
            img = Image.open(result.path).convert("RGB")
            
            # Original
            axes[i][0].imshow(img)
            axes[i][0].set_title(f"#{i+1} Score: {result.combined_score:.3f}\nCLIP: {result.clip_score:.3f}, YOLO: {result.yolo_score:.3f}")
            axes[i][0].axis('off')
            
            # With detections
            img_det = yolo_detector.draw_detections(img, result.yolo_detections)
            axes[i][1].imshow(img_det)
            axes[i][1].set_title(f"Detections: {result.num_detections} ({result.num_interesting} interesting)")
            axes[i][1].axis('off')
            
        except Exception as e:
            axes[i][0].text(0.5, 0.5, f"Error loading image", ha='center', va='center')
            axes[i][1].text(0.5, 0.5, f"Error loading image", ha='center', va='center')
    
    plt.tight_layout()
    plt.show()


# Example usage (uncomment when you have results):
# save_results(camera_results, RESULTS_DIR, "camera_top_200")
# visualize_top_results(camera_results, n=10)

## 14. Process Local Images (Alternative)

If you already have images extracted locally, use this to process them directly.

In [ ]:
def process_local_directory(image_dir: Path, top_k: int = 200, recursive: bool = True) -> List[ImageResult]:
    """
    Process all images in a local directory.
    
    Args:
        image_dir: Path to directory containing images
        top_k: Number of top images to return
        recursive: Whether to search subdirectories
    """
    image_dir = Path(image_dir)
    
    if not image_dir.exists():
        print(f"Error: Directory not found: {image_dir}")
        return []
    
    image_paths = []
    
    if recursive:
        for root, dirs, files in os.walk(image_dir):
            for f in files:
                ext = os.path.splitext(f)[1].lower()
                if ext in IMAGE_EXTENSIONS:
                    image_paths.append(Path(root) / f)
    else:
        for f in image_dir.iterdir():
            if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS:
                image_paths.append(f)
    
    print(f"Found {len(image_paths)} images in {image_dir}")
    
    if not image_paths:
        return []
    
    return process_images(image_paths, clip_scorer, yolo_detector, top_k=top_k)


def process_any_dataset(path: str, top_k: int = 200) -> List[ImageResult]:
    """
    Universal function to process any dataset.
    
    Automatically detects if the path is:
    - A directory of images
    - An archive file (.7z, .zip, .tar, .tar.gz)
    - A split archive (.7z.001)
    
    Args:
        path: Path to directory or archive
        top_k: Number of top images to return
    """
    path = Path(path)
    
    if not path.exists():
        print(f"Error: Path not found: {path}")
        return []
    
    # Check if it's a directory
    if path.is_dir():
        print(f"Detected: Directory")
        return process_local_directory(path, top_k=top_k)
    
    # Check if it's an archive
    name = path.name.lower()
    if any(ext in name for ext in ['.7z', '.zip', '.tar', '.tgz']):
        print(f"Detected: Archive file")
        return process_local_archive(path, clip_scorer, yolo_detector, top_k=top_k)
    
    print(f"Error: Unsupported path type: {path}")
    return []


# ============================================================
# EASY USAGE - Just change this path to your dataset!
# ============================================================

# Set your dataset path here (directory or archive):
DATASET_PATH = "/path/to/your/images"  # <-- CHANGE THIS!

# Example paths:
# DATASET_PATH = "/media/priyanshu/2TB SSD/Lockheed2026/underwater_images"
# DATASET_PATH = "./ranked_dataset/temp/camera.7z.001"
# DATASET_PATH = "./my_downloaded_images.zip"

print(f"Dataset path: {DATASET_PATH}")
print(f"Path exists: {Path(DATASET_PATH).exists()}")

# Uncomment to run:
# results = process_any_dataset(DATASET_PATH, top_k=TOP_K)
# if results:
#     save_results(results, RESULTS_DIR, "my_dataset_results")
#     visualize_top_results(results, n=10)

In [ ]:
# Debug the API response
import requests

doi = "doi:10.7910/DVN/VZD5S6"
base_url = "https://dataverse.harvard.edu"
url = f"{base_url}/api/datasets/:persistentId/?persistentId={doi}"

print(f"Testing URL: {url}\n")

response = requests.get(url)
print(f"Status Code: {response.status_code}")
print(f"Content-Type: {response.headers.get('content-type', 'N/A')}")
print(f"Content-Length: {len(response.text)}")
print(f"\nFirst 500 chars of response:")
print(response.text[:500])
print(f"\n---\nResponse headers:")
for key, value in response.headers.items():
    print(f"  {key}: {value}")

In [6]:
# Comprehensive API diagnostic
import requests
import json

print("=" * 60)
print("DATAVERSE API DIAGNOSTIC")
print("=" * 60)

doi = "doi:10.7910/DVN/VZD5S6"
base_url = "https://dataverse.harvard.edu"

# Test 1: Check if we can reach Harvard Dataverse at all
print("\n[Test 1] Can we reach Harvard Dataverse?")
try:
    response = requests.get(base_url, timeout=10)
    print(f"  ✓ Base URL reachable: {response.status_code}")
except Exception as e:
    print(f"  ✗ Cannot reach base URL: {e}")
    print(f"  This suggests a network/firewall issue")

# Test 2: Try the API endpoint with the DOI
print(f"\n[Test 2] Testing API endpoint with DOI")
api_url = f"{base_url}/api/datasets/:persistentId/?persistentId={doi}"
print(f"  URL: {api_url}")

try:
    response = requests.get(api_url, timeout=30, headers={'User-Agent': 'Mozilla/5.0'})
    print(f"  Status: {response.status_code}")
    print(f"  Content-Type: {response.headers.get('content-type', 'N/A')}")
    print(f"  Content-Length: {len(response.text)} bytes")
    
    if response.text:
        print(f"\n  First 300 characters of response:")
        print(f"  {response.text[:300]}")
        
        # Try to parse as JSON
        try:
            data = response.json()
            print(f"\n  ✓ Valid JSON response!")
            print(f"  Keys: {list(data.keys())}")
            if 'data' in data:
                print(f"  ✓ Has 'data' key")
                if 'latestVersion' in data.get('data', {}):
                    print(f"  ✓ Has 'latestVersion'")
                    files = data['data']['latestVersion'].get('files', [])
                    print(f"  ✓ Found {len(files)} files")
        except json.JSONDecodeError as e:
            print(f"\n  ✗ Not valid JSON: {e}")
            print(f"  Response might be HTML error page")
    else:
        print(f"  ✗ Empty response body")
        
except requests.exceptions.Timeout:
    print(f"  ✗ Request timed out")
except Exception as e:
    print(f"  ✗ Error: {e}")

# Test 3: Try alternative API endpoint format
print(f"\n[Test 3] Try alternative URL format")
alt_url = f"{base_url}/api/datasets/:persistentId?persistentId={doi}"  # No trailing slash
print(f"  URL: {alt_url}")

try:
    response = requests.get(alt_url, timeout=30, headers={'User-Agent': 'Mozilla/5.0'})
    print(f"  Status: {response.status_code}")
    print(f"  Response length: {len(response.text)} bytes")
    if response.text:
        print(f"  First 200 chars: {response.text[:200]}")
except Exception as e:
    print(f"  Error: {e}")

# Test 4: Check the dataset page directly
print(f"\n[Test 4] Dataset webpage")
web_url = f"{base_url}/dataset.xhtml?persistentId={doi}"
print(f"  URL: {web_url}")
print(f"  Try opening this in your browser to verify the dataset exists")

try:
    response = requests.get(web_url, timeout=30, headers={'User-Agent': 'Mozilla/5.0'})
    print(f"  Status: {response.status_code}")
    if "harvard" in response.text.lower() and "dataverse" in response.text.lower():
        print(f"  ✓ Page loads and contains 'harvard' and 'dataverse'")
    else:
        print(f"  ✗ Page might not be the expected dataset page")
except Exception as e:
    print(f"  Error: {e}")

print("\n" + "=" * 60)

DATAVERSE API DIAGNOSTIC

[Test 1] Can we reach Harvard Dataverse?
  ✗ Cannot reach base URL: HTTPSConnectionPool(host='dataverse.harvard.edu', port=443): Max retries exceeded with url: / (Caused by ConnectTimeoutError(<HTTPSConnection(host='dataverse.harvard.edu', port=443) at 0x78d5e916c2b0>, 'Connection to dataverse.harvard.edu timed out. (connect timeout=10)'))
  This suggests a network/firewall issue

[Test 2] Testing API endpoint with DOI
  URL: https://dataverse.harvard.edu/api/datasets/:persistentId/?persistentId=doi:10.7910/DVN/VZD5S6
  ✗ Request timed out

[Test 3] Try alternative URL format
  URL: https://dataverse.harvard.edu/api/datasets/:persistentId?persistentId=doi:10.7910/DVN/VZD5S6
  Error: HTTPSConnectionPool(host='dataverse.harvard.edu', port=443): Max retries exceeded with url: /api/datasets/:persistentId?persistentId=doi:10.7910/DVN/VZD5S6 (Caused by ConnectTimeoutError(<HTTPSConnection(host='dataverse.harvard.edu', port=443) at 0x78d5e9aab550>, 'Connection to dat

In [7]:
# Check if you have images elsewhere on your system
import os
from pathlib import Path

# Common locations to check
possible_locations = [
    "/media/priyanshu/2TB SSD/Lockheed2026",
    str(Path.home() / "Downloads"),
    str(Path.home() / "Documents"),
    "./",
]

print("Searching for image directories...")
for base_path in possible_locations:
    if not os.path.exists(base_path):
        continue
    
    for root, dirs, files in os.walk(base_path):
        # Check depth to avoid going too deep
        depth = root[len(base_path):].count(os.sep)
        if depth > 3:
            continue
            
        # Look for directories with many images
        image_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif', '.tiff'))]
        if len(image_files) > 10:
            print(f"\nFound: {root}")
            print(f"  Images: {len(image_files)}")
            print(f"  Sample: {image_files[:3]}")

Searching for image directories...


In [8]:
# Example: Configure proxy if your network requires it
# Uncomment and modify these lines if you need proxy access:

# import os
# os.environ['HTTP_PROXY'] = 'http://your-proxy:port'
# os.environ['HTTPS_PROXY'] = 'http://your-proxy:port'

# Alternative: Use a VPN or download on a different network
# Then transfer the files to this machine for processing

print("Network configuration ready")

Network configuration ready
